In [2]:
import torch 
import torch.nn as nn
import torch.nn.functional as F 
import math

toy dataset 

In [3]:
sentences = [
    ["I", "love", "AI"],
    ["AI", "learns", "fast"],
    ["You", "love", "Python"]
]

vocab = {
    "<PAD>": 0,
    "I": 1,
    "love": 2,
    "AI": 3,
    "learns": 4,
    "fast": 5,
    "You": 6,
    "Python": 7
}


encode

In [4]:
encoded=[]

In [5]:
for sentence in sentences:
       ids=[vocab[word] for word in sentence]
       encoded.append(ids)
x_ids=torch.tensor(encoded)

In [6]:
print(x_ids)
print(x_ids.shape)

tensor([[1, 2, 3],
        [3, 4, 5],
        [6, 2, 7]])
torch.Size([3, 3])


torch size niba batch size er jonno 

In [8]:
vocab_size=len(vocab)
embed_dim=8
embedding=nn.Embedding(vocab_size, embed_dim)
x=embedding(x_ids)
print(x.shape)


torch.Size([3, 3, 8])


In [11]:
class selfHeadAttention(nn.Module):
       def __init__(self,embed_dim):
              super().__init__()
              self.embed_dim=embed_dim

              self.W_q=nn.Linear(embed_dim,embed_dim)
              self.W_k=nn.Linear(embed_dim,embed_dim)
              self.W_v=nn.Linear(embed_dim,embed_dim)
       def forward(self,x):
              Q=self.W_q(x)
              K=self.W_k(x)
              V=self.W_v(x)

              scores=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(self.embed_dim)
              attention_weights=F.softmax(scores,dim=-1)
              output=torch.matmul(attention_weights,V)

              return output, attention_weights



In [12]:
single_head=selfHeadAttention(embed_dim)
self_head_output,self_weights=single_head(x)
print("Single Head Output Shape:", self_head_output.shape)
print("Attention Weights Shape:", self_weights.shape)
print(self_weights)

Single Head Output Shape: torch.Size([3, 3, 8])
Attention Weights Shape: torch.Size([3, 3, 3])
tensor([[[0.3257, 0.3390, 0.3353],
         [0.3399, 0.3123, 0.3479],
         [0.3633, 0.3191, 0.3177]],

        [[0.3039, 0.3892, 0.3069],
         [0.3247, 0.5198, 0.1554],
         [0.2990, 0.3972, 0.3038]],

        [[0.3571, 0.2965, 0.3464],
         [0.3840, 0.3808, 0.2352],
         [0.4832, 0.3522, 0.1646]]], grad_fn=<SoftmaxBackward0>)


In [13]:
single_head=selfHeadAttention(embed_dim)
single_output,self_weights=single_head(x)
print("Single Head Output Shape:", single_output.shape)
print("Attention Weights Shape:", self_weights.shape)
print(self_weights)

Single Head Output Shape: torch.Size([3, 3, 8])
Attention Weights Shape: torch.Size([3, 3, 3])
tensor([[[0.1902, 0.4685, 0.3413],
         [0.4155, 0.1967, 0.3878],
         [0.3871, 0.2834, 0.3295]],

        [[0.3410, 0.2672, 0.3918],
         [0.4454, 0.2285, 0.3261],
         [0.2618, 0.5577, 0.1805]],

        [[0.3511, 0.3330, 0.3159],
         [0.4988, 0.1950, 0.3062],
         [0.6830, 0.1206, 0.1964]]], grad_fn=<SoftmaxBackward0>)


multi head attention 

In [18]:
class MultiHeadAttention(nn.Module):
       def __init__(self,embed_dim,num_heads):
              super().__init__()
              assert embed_dim % num_heads==0 
              self.embed_dim=embed_dim
              self.num_heads=num_heads 
              self.head_dim=embed_dim//num_heads

              self.W_q=nn.Linear(embed_dim,embed_dim)
              self.W_k=nn.Linear(embed_dim,embed_dim)
              self.W_v=nn.Linear(embed_dim,embed_dim)
              self.out_proj=nn.Linear(embed_dim,embed_dim)

       def forward(self,x):
              batch_size,seq_len,embed_dim=x.shape
              Q=self.W_q(x)
              K=self.W_k(x)
              V=self.W_v(x)

              Q=Q.view(batch_size,seq_len,self.num_heads,self.head_dim).transpose(1,2)
              K=K.view(batch_size,seq_len,self.num_heads,self.head_dim).transpose(1,2)
              V=V.view(batch_size,seq_len,self.num_heads,self.head_dim).transpose(1,2)
              scores=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(self.head_dim)
              attention_weights=F.softmax(scores,dim=-1)
              attention_output=torch.matmul(attention_weights,V)
              attention_output=attention_output.transpose(1,2).contiguous().view(batch_size,seq_len,self.embed_dim)
              output=self.out_proj(attention_output)
              return output, attention_weights

tesnor swap 

In [19]:
import torch 
Q = torch.randn(3,8)
K=torch.randn(3,8)
print(Q.shape)
print(K.shape)

torch.Size([3, 8])
torch.Size([3, 8])


In [20]:
multi_head = MultiHeadAttention(embed_dim=8, num_heads=2)

multi_output, multi_weights = multi_head(x)

print("Multi Head Output Shape:", multi_output.shape)
print("Multi Head Attention Shape:", multi_weights.shape)

Multi Head Output Shape: torch.Size([3, 3, 8])
Multi Head Attention Shape: torch.Size([3, 2, 3, 3])


using minst dataset

In [24]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import MNIST 
from torchvision import transforms 
from torch.utils.data import DataLoader 
import math 

In [25]:
transform=transforms.ToTensor()
dataset=MNIST(root="./data",train=True,transform=transform,download=True)
loader=DataLoader(dataset,batch_size=4,shuffle=True)
images,labels=next(iter(loader))
print("Batch Images Shape:", images.shape)
print("Batch Labels Shape:", labels.shape)

100%|██████████| 9.91M/9.91M [00:12<00:00, 777kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 87.1kB/s]
100%|██████████| 1.65M/1.65M [00:06<00:00, 275kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.38MB/s]

Batch Images Shape: torch.Size([4, 1, 28, 28])
Batch Labels Shape: torch.Size([4])


multihead er kaj suru 

In [27]:
b=images.shape[0]
b

4

In [28]:
x=images.view(b,28*28,1)
print("Reshaped Input Shape:", x.shape)

Reshaped Input Shape: torch.Size([4, 784, 1])


dimensional vector banamu 

In [29]:
embed_dim=32 
pro=nn.Linear(1,embed_dim)
x=pro(x)
print(x.shape)

torch.Size([4, 784, 32])


In [30]:
x[0]

tensor([[ 0.5867,  0.4167,  0.2663,  ...,  0.6538, -0.3141, -0.8526],
        [ 0.5867,  0.4167,  0.2663,  ...,  0.6538, -0.3141, -0.8526],
        [ 0.5867,  0.4167,  0.2663,  ...,  0.6538, -0.3141, -0.8526],
        ...,
        [ 0.5867,  0.4167,  0.2663,  ...,  0.6538, -0.3141, -0.8526],
        [ 0.5867,  0.4167,  0.2663,  ...,  0.6538, -0.3141, -0.8526],
        [ 0.5867,  0.4167,  0.2663,  ...,  0.6538, -0.3141, -0.8526]],
       grad_fn=<SelectBackward0>)

multi head attention

In [ ]:
class multiheadattention(nn.Module):
       def __init__(self,embed_dim,num_heads):
              assert embed_dim % num_heads==0
              super().__init__()
              self.embed_dim = embed_dim
              self.num_heads = num_heads
              self.head_dim = embed_dim // num_heads
              self.W_q = nn.Linear(embed_dim, embed_dim)
              self.W_k = nn.Linear(embed_dim, embed_dim)
              self.W_v = nn.Linear(embed_dim, embed_dim)
              self.out_proj = nn.Linear(embed_dim, embed_dim)
       def forward(self,x):
              B,S,E=x.shape
              Q=self.W_q(x)
              K=self.W_k(x)
              V=self.W_v(x)
              print("Q Shape:", Q.shape)
              print("K Shape:", K.shape)
              print("V Shape:", V.shape)
              Q=Q.view(B,S,self.num_heads,self.head_dim).transpose(1,2)
              K=K.view(B,S,self.num_heads,self.head_dim).transpose(1,2)
              V=V.view(B,S,self.num_heads,self.head_dim).transpose(1,2)
              print("Q Reshaped Shape:", Q.shape)
              print("K Reshaped Shape:", K.shape)
              print("V Reshaped Shape:", V.shape)
              scores=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(self.head_dim)
              print("Scores Shape:", scores.shape)
              attention_weights=F.softmax(scores,dim=-1)
              print("Attention Weights Shape:", attention_weights.shape)
              attention_output=torch.matmul(attention_weights,V)
              print("Attention Output Shape:", attention_output.shape)
              attention_output=attention_output.transpose(1,2).contiguous().view(B,S,self.embed_dim)
              output=self.out_proj(attention_output)
              return output, attention_weights